# 03 - NumPy Advanced: Memory, Strides, Structured Arrays, Linear Algebra

**Prerequisite:** notebooks 01-02.
**This notebook:** what is happening underneath - memory order, strides, sliding windows, structured arrays (the closest thing to a table of POJOs), datetime64, linear algebra, and `einsum`.

This is the material that separates "I can use NumPy" from "I know why my code is slow".

In [ ]:
import numpy as np
rng = np.random.default_rng(11)

## 1. Memory order: C vs Fortran

A 2-D array is one flat block. The **order** decides how rows and columns map onto it.

- **C order** (row-major, the default): the last axis varies fastest - a whole row sits together. Same as Java's `int[][]` rows, and same as C.
- **F order** (column-major): the first axis varies fastest - a whole column sits together. Same as Fortran, MATLAB, R.

This matters because CPUs read memory in cache lines. Walking along the contiguous direction is several times faster than jumping across it.

In [ ]:
m = np.arange(6).reshape(2, 3)              # C order by default
f = np.asfortranarray(m)

print("same values:\n", m)
print()
print("C order flat memory:", m.ravel(order="K"))
print("F order flat memory:", f.ravel(order="K"))
print()
print("m flags: C_CONTIGUOUS =", m.flags["C_CONTIGUOUS"], ", F_CONTIGUOUS =", m.flags["F_CONTIGUOUS"])
print("f flags: C_CONTIGUOUS =", f.flags["C_CONTIGUOUS"], ", F_CONTIGUOUS =", f.flags["F_CONTIGUOUS"])

In [ ]:
import time

big_c = np.ones((5000, 5000))
big_f = np.asfortranarray(big_c)

def best_of(fn, n=3):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return round(min(times), 4)

print("copy a C-contiguous array to C layout :", best_of(lambda: big_c.ravel().copy()), "s")
print("copy an F-contiguous array to C layout:", best_of(lambda: np.ascontiguousarray(big_f)), "s  <- same bytes, wrong direction")

The second copy reads memory in a jumping pattern and is several times slower for identical data. That is pure cache behaviour.

A caution about reductions, because the intuition trips people up: `arr.sum(axis=0)` on a C-order array is usually *faster* than `arr.sum(axis=1)`, even though axis 1 is the contiguous one. Summing down columns accumulates whole contiguous rows into an output row, which the CPU can do with SIMD instructions; summing across a row is a horizontal reduction that cannot. So measure before optimising - the rule is "keep data contiguous in the direction you traverse it", not "always use axis=1".

## 2. Strides - how NumPy fakes dimensions

`strides` tells NumPy how many **bytes** to step to move one position along each axis. This single piece of metadata is how views, transposes and reshapes are free.

In [ ]:
m = np.arange(12, dtype=np.int64).reshape(3, 4)
print("shape  :", m.shape)
print("strides:", m.strides, " -> step 32 bytes for the next row, 8 bytes for the next column")
print()
print("Transposing just swaps the strides - zero data movement:")
print("m.T shape  :", m.T.shape)
print("m.T strides:", m.T.strides)
print("shares memory?", np.shares_memory(m, m.T))

In [ ]:
# A slice is just an offset plus adjusted strides
s = m[:, 1::2]
print("m[:, 1::2] shape :", s.shape)
print("strides          :", s.strides, " -> column step is now 16 bytes (skipping one)")
print("still a view?    ", s.base is not None)

### Sliding windows for free

`sliding_window_view` builds a windowed view using strides alone. No data is copied, so a rolling window over 10 million points costs nothing in memory.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

prices = np.array([10., 12., 11., 15., 14., 18., 17.])
windows = sliding_window_view(prices, window_shape=3)

print("windows shape:", windows.shape)
print(windows)
print()
print("3-point moving average:", windows.mean(axis=1).round(2))
print("copies made:", 0, "- windows.base is prices?", windows.base is not None)

> **Warning:** a strided view can overlap itself. Never write into a `sliding_window_view` - the same memory cell appears in several windows and you will corrupt data. Read only.

## 3. Structured arrays - a table in one array

A structured dtype gives each element named fields of different types. This is the closest NumPy gets to `List<Order>` in Java, or a database row.

In practice you would reach for Pandas instead, but you meet structured arrays when reading binary formats, `np.genfromtxt` output, and some scientific file formats.

In [ ]:
order_dtype = np.dtype([
    ("order_id", "i8"),        # int64      -> long
    ("customer", "U12"),       # unicode 12 -> String
    ("amount",   "f8"),        # float64    -> double
    ("paid",     "?"),         # bool       -> boolean
])

orders = np.array([
    (1001, "acme",   250.50, True),
    (1002, "globex", 99.99,  False),
    (1003, "acme",   1200.0, True),
], dtype=order_dtype)

print(orders)
print()
print("one field (a column):", orders["amount"])
print("one element (a row) :", orders[0])
print("field of one row    :", orders[0]["customer"])
print()
print("WHERE paid AND amount > 100:", orders[orders["paid"] & (orders["amount"] > 100)]["order_id"])
print("SUM(amount)                :", orders["amount"].sum())

## 4. datetime64 and timedelta64

NumPy stores dates as a 64-bit integer count of a fixed unit since the Unix epoch. The unit is part of the dtype: `datetime64[D]` is days, `datetime64[ns]` is nanoseconds (what Pandas uses).

Java analogy: `datetime64[ns]` is `Instant`, `timedelta64` is `Duration`. There is **no timezone** - it is always naive. Timezone handling lives in Pandas.

In [ ]:
d = np.array(["2026-01-15", "2026-03-01", "2026-09-14"], dtype="datetime64[D]")
print("dates   :", d)
print("dtype   :", d.dtype)
print("as int  :", d.astype("int64"), " <- days since 1970-01-01")
print()
print("date arithmetic:", d + np.timedelta64(30, "D"))
print("differences    :", np.diff(d))
print("to nanoseconds :", d.astype("datetime64[ns]")[0])

In [ ]:
print("now (truncated to seconds):", np.datetime64("now", "s"))
print("business days between      :", np.busday_count("2026-09-01", "2026-09-14"))
print("is 2026-09-13 a weekday?   :", np.is_busday("2026-09-13"))

## 5. Linear algebra

`np.linalg` wraps LAPACK/BLAS - the same battle-tested Fortran libraries every numerical stack uses.

In [ ]:
A_ = np.array([[2., 1.],
               [1., 3.]])
b_ = np.array([5., 10.])

print("determinant :", np.linalg.det(A_).round(6))
print("inverse     :\n", np.linalg.inv(A_).round(4))
print()
print("solve A x = b:", np.linalg.solve(A_, b_))
print("check A @ x  :", A_ @ np.linalg.solve(A_, b_))

> Use `np.linalg.solve(A, b)`, never `np.linalg.inv(A) @ b`. Solving directly is faster and numerically more stable - the same reason you use a prepared statement instead of building SQL by hand.

In [ ]:
vals, vecs = np.linalg.eig(A_)
print("eigenvalues :", vals.round(4))
print()
print("matrix norm :", np.linalg.norm(A_).round(4))
print("rank        :", np.linalg.matrix_rank(A_))
print("trace       :", np.trace(A_))

In [ ]:
# Least squares: fit y = m*x + c to noisy data
x = np.arange(20, dtype=float)
y = 3.5 * x + 7 + rng.normal(0, 2, 20)

design = np.column_stack([x, np.ones_like(x)])          # columns: [x, 1]
(m_, c_), *_ = np.linalg.lstsq(design, y, rcond=None)
print("fitted slope:", round(float(m_), 3), " intercept:", round(float(c_), 3), " (true: 3.5 and 7)")

## 6. `einsum` - one function for most tensor work

`np.einsum` uses index notation. Repeated indices are summed over; indices left of `->` that do not appear on the right are summed away.

It is worth learning because it makes intent explicit and often avoids intermediate arrays.

In [ ]:
a = np.arange(1, 5)
b = np.arange(5, 9)
M = np.arange(1, 7).reshape(2, 3)
N = np.arange(1, 7).reshape(3, 2)

print("dot product        einsum('i,i->')    :", np.einsum("i,i->", a, b),        " == ", a @ b)
print("outer product      einsum('i,j->ij')  shape:", np.einsum("i,j->ij", a, b).shape)
print("matrix multiply    einsum('ij,jk->ik'):\n", np.einsum("ij,jk->ik", M, N))
print("transpose          einsum('ij->ji')   shape:", np.einsum("ij->ji", M).shape)
print("row sums           einsum('ij->i')    :", np.einsum("ij->i", M))
print("trace              einsum('ii->')     :", np.einsum("ii->", np.eye(4)))

## 7. Vectorizing awkward logic

Sometimes the operation does not map to a ufunc. Options, best first:

1. Rewrite with masks / `np.where` / `np.select`.
2. Use `np.add.at`-style **unbuffered** operations for scattered updates.
3. `np.apply_along_axis` - convenience only, still a Python loop.
4. `np.vectorize` - convenience only, **not** a speed-up. The docs say so explicitly.

In [ ]:
# np.add.at: scatter-add. Plain fancy-index += silently drops duplicate positions.
counts = np.zeros(5, dtype=int)
idx = np.array([0, 1, 1, 1, 4])

counts[idx] += 1
print("with += :", counts, " <- WRONG, index 1 counted once")

counts = np.zeros(5, dtype=int)
np.add.at(counts, idx, 1)
print("add.at  :", counts, " <- correct, this is a GROUP BY count")

In [ ]:
# np.bincount is the fast way to do the same thing for small integer keys
print("bincount:", np.bincount(idx, minlength=5))

# weighted: sum an amount per key - a full GROUP BY key, SUM(amount)
keys    = np.array([0, 1, 1, 2, 0])
amounts = np.array([10., 5., 7., 3., 2.])
print("group sums by key:", np.bincount(keys, weights=amounts))

## 8. Memory hygiene

| Habit | Why |
|---|---|
| pick the smallest dtype that fits (`int32`, `float32`) | halves memory and doubles cache efficiency |
| use in-place ops (`a += b`, `out=`) in hot loops | avoids allocating a new array each step |
| avoid `dtype=object` | every element is a boxed Python object; you lose all the speed |
| avoid chained temporaries on huge arrays | `(a + b) * c - d` allocates 3 intermediates |
| check `arr.nbytes` and `np.shares_memory(x, y)` | tells you what is really allocated and shared |

In [ ]:
import sys

a = rng.normal(size=1_000_000)
print("float64 array, 1M values:", a.nbytes // 1024, "KB")
print("float32 array, 1M values:", a.astype(np.float32).nbytes // 1024, "KB")

obj = np.array(a[:100_000], dtype=object)
pointers = obj.nbytes
boxed = sum(sys.getsizeof(v) for v in obj[:1000]) / 1000 * obj.size
print("object  array, 0.1M vals:", int(pointers + boxed) // 1024, "KB  <- pointers PLUS a Python float object each")
print("                          that is roughly", round((pointers + boxed) / 100_000 / (a.nbytes / 1_000_000), 1), "x the memory per value")

## 9. Exercises

**E1.** Given `x = np.arange(1, 16)`, compute the 4-point moving sum using `sliding_window_view`.

**E2.** Build a structured array of 3 employees (id, name, dept, salary) and compute the average salary per department using `np.unique` + a loop over departments only (2-3 departments, so the loop is over groups, not rows).

**E3.** Use `einsum` to compute the row-wise dot product of two `(4,3)` matrices (result shape `(4,)`).

**E4.** `keys = np.array([2,0,1,2,2,1])`, `vals = np.array([1.,2.,3.,4.,5.,6.])`. Compute the sum and the count per key without any Python loop.

In [ ]:
# Solutions
from numpy.lib.stride_tricks import sliding_window_view

x = np.arange(1, 16)
print("E1:", sliding_window_view(x, 4).sum(axis=1))
print()

emp_dt = np.dtype([("id", "i4"), ("name", "U10"), ("dept", "U10"), ("salary", "f8")])
emp = np.array([(1, "asha", "eng", 120.), (2, "biju", "eng", 140.), (3, "chitra", "sales", 90.)], dtype=emp_dt)
for d in np.unique(emp["dept"]):
    print("E2:", d, "->", emp["salary"][emp["dept"] == d].mean())
print()

P = rng.normal(size=(4, 3)); Q = rng.normal(size=(4, 3))
print("E3:", np.einsum("ij,ij->i", P, Q).round(4))
print("   same as (P*Q).sum(axis=1):", (P * Q).sum(axis=1).round(4))
print()

keys = np.array([2, 0, 1, 2, 2, 1]); vals = np.array([1., 2., 3., 4., 5., 6.])
print("E4 sums  :", np.bincount(keys, weights=vals))
print("E4 counts:", np.bincount(keys))
print("E4 means :", np.bincount(keys, weights=vals) / np.bincount(keys))

## Recap

- **Strides** are the whole trick: views, transposes, reshapes and sliding windows are metadata changes, not data copies.
- **C order** means rows are contiguous. Reduce along the contiguous axis when you can.
- **Structured arrays** give you named typed fields - a table, but Pandas does this better.
- `datetime64` is an integer count since the epoch, with no timezone.
- `np.linalg.solve` beats `inv`; `einsum` expresses most tensor contractions in one line.
- `np.add.at` / `np.bincount` are the group-by primitives; plain `arr[idx] += 1` loses duplicates.

**Next:** `04_pandas_foundations.ipynb` - Series, DataFrame, and the index.